# Recommendations demo

**Goal:** for a given input item, retrieve and compare the recommendations
produced by all three models in this project -- TTN ("Complete the Look",
complementary products), SigLIP2 ("Visually Similar Products",
substitutes), and Popularity ("Trending in Category") -- using each
model's current **champion**, as recorded in that model's own
`champion_selection_*.json`, not an arbitrary version chosen ad hoc here.

This is a different kind of model access than the live API
(`api.py`)/demo page (`demo/index.html`) uses: there, every request is a
plain SQLite lookup against work already done offline. Here:

- **TTN's and Popularity's champions already have precomputed
  recommendation tables on disk** (`complements.parquet` /
  `popularity.parquet`, written by each model's own
  `generate_recommendations.py` / `build_popularity.py`) -- for those two,
  this notebook just reads the existing table, the same as the live API
  would, no model execution involved.
- **SigLIP2's champion (the imported `pilot_200k` adapter) has no
  precomputed recommendation table yet** -- nothing in this repo has run
  it across the catalogue and saved the result. So for SigLIP2 specifically,
  this notebook loads the live adapter checkpoint, as a first step toward
  running it over the full catalogue and storing those results, the same
  way the other two models already work.

In [9]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# demo/recommendations_demo.ipynb's kernel cwd is this notebook's own folder
# (demo/) when opened normally -- walk up to the repo root so the SigLIP2
# package import below and every path resolved later are correct regardless
# of how the kernel was launched.
ROOT = Path.cwd()
if not (ROOT / "SigLIP2").exists():
    ROOT = ROOT.parent
assert (ROOT / "SigLIP2").exists(), f"couldn't find the repo root from {Path.cwd()}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from SigLIP2.adapter import load_adapter

print(f"repo root: {ROOT}")

repo root: /Users/lazr/PycharmProjects/RecSystem


In [ ]:
# Read each model's own champion-selection record. Each file is hand-edited,
# not written by any script -- see this session's discussion for why the
# schemas differ: TTN's candidates are always {snapshot_id, version_id}
# (version numbering restarts per snapshot, so both fields are required to
# identify one model); SigLIP2's aren't uniformly snapshot-scoped (the
# imported pilot_200k checkpoint isn't tied to any --window-days
# snapshot), so it uses a direct checkpoint_path instead; Popularity's is
# now just a note -- it no longer takes --snapshot at all (categories come
# from df_features.pkl directly, not TTN's co-purchase-derived item list),
# so there's only one universal popularity.parquet, nothing to select.
CHAMPIONS = {
    "ttn": json.loads((ROOT / "TTN" / "champion_selection_ttn.json").read_text()),
    "siglip2": json.loads((ROOT / "SigLIP2" / "champion_selection_siglip.json").read_text()),
}
for name, c in CHAMPIONS.items():
    print(name, {k: v for k, v in c.items() if k != "_note"})

# --- TTN: champion already has precomputed recommendations -- just resolve
# the path, no model loading needed.
ttn_snapshot = CHAMPIONS["ttn"]["snapshot_id"]
ttn_version = CHAMPIONS["ttn"]["version_id"]
ttn_complements_path = (ROOT / "TTN" / "generated" / ttn_snapshot
                         / "recommendations" / "complements.parquet")
assert ttn_complements_path.exists(), f"missing: {ttn_complements_path}"
print(f"\nTTN champion recommendations -> {ttn_complements_path.relative_to(ROOT)}")

# --- Popularity: no snapshot, no version -- one universal file, always
# at this fixed path.
popularity_path = ROOT / "Popularity" / "generated" / "recommendations" / "popularity.parquet"
assert popularity_path.exists(), f"missing: {popularity_path}"
print(f"Popularity recommendations -> {popularity_path.relative_to(ROOT)}")

# --- SigLIP2: champion has no precomputed recommendations yet -- load the
# live adapter checkpoint instead.
siglip_checkpoint_path = ROOT / CHAMPIONS["siglip2"]["checkpoint_path"]
assert siglip_checkpoint_path.exists(), f"missing: {siglip_checkpoint_path}"
siglip_adapter = load_adapter(checkpoint_path=siglip_checkpoint_path)
print(f"SigLIP2 champion adapter loaded ({type(siglip_adapter).__name__}) "
      f"from {siglip_checkpoint_path.relative_to(ROOT)}")

## Compare TTN's two recommendation approaches

Both files below come from the exact same trained champion model
(`ttn_snapshot`/`ttn_version` above) -- the only thing that differs is the
GENERATION-time merge rule across a query's multiple licensed target
categories:

- **`complements.parquet`** (the "20 most relevant items") -- pools every
  licensed category's candidates into one flat pool and keeps the single
  overall top-20 by score. Can let one category dominate entirely if it
  happens to score higher -- verified this session on a real item: its
  flat top-20 was 20/20 from a single category, ignoring 4 other licensed
  categories with real co-purchase evidence behind them.
- **`complements_proportional.parquet`** (the "weighted 20 items") --
  splits the 20 slots across licensed categories in proportion to each
  category's co-purchase evidence (`edges`, from
  `complementary_categories.pkl`), then ranks independently *within* each
  category -- guarantees every licensed category with a nonzero share
  gets represented, at the cost of sometimes filling a slot with a weaker
  same-category match instead of a stronger one from an under-allocated
  category.

In [ ]:
import html as _html
import json

from IPython.display import HTML, display

# Resolve the proportional file's path too -- same champion snapshot/version
# as ttn_complements_path above, just a different generation script's output
# (TTN/generate_recommendations_proportional.py instead of
# TTN/generate_recommendations.py).
ttn_proportional_path = (ROOT / "TTN" / "generated" / ttn_snapshot
                          / "recommendations" / "complements_proportional.parquet")
assert ttn_proportional_path.exists(), f"missing: {ttn_proportional_path}"
print(f"TTN proportional recommendations -> {ttn_proportional_path.relative_to(ROOT)}")

ttn_flat = pd.read_parquet(ttn_complements_path)
ttn_proportional = pd.read_parquet(ttn_proportional_path)

# asin -> product image url. Reads the small precomputed cache
# (data_creation/build_image_url_cache.py) rather than df_features.pkl
# directly -- df_features.pkl is 1.13M rows x ~90 columns, so loading it
# just to reach its 2 image-url columns took ~7.4s on its own; this cache
# has only what's needed and loads in ~0.15s. Regenerate it (python
# data_creation/build_image_url_cache.py) if it's missing or stale.
image_url_cache_path = ROOT / "data" / "asin_image_urls.json"
assert image_url_cache_path.exists(), (
    f"missing: {image_url_cache_path} -- run "
    f"'python data_creation/build_image_url_cache.py' first")
image_by_asin = json.loads(image_url_cache_path.read_text())


def _card(asin: str, caption: str = "", score: float = None) -> str:
    """One product tile: picture if available, else a grey placeholder --
    about half of items have no image at all, so this has to degrade
    gracefully rather than assume one exists."""
    url = image_by_asin.get(asin)
    pic = (f'<img src="{_html.escape(str(url))}" style="width:110px;height:110px;'
           f'object-fit:contain;background:#fff;border-radius:4px">'
           if isinstance(url, str) else
           '<div style="width:110px;height:110px;background:#eee;border-radius:4px;'
           'display:flex;align-items:center;justify-content:center;color:#999;'
           'font-size:10px">no image</div>')
    head = f'<div style="font-size:10px;color:#c60">{_html.escape(caption)}</div>' if caption else ""
    sc = f'<div style="font-size:10px;color:#888">{score:+.3f}</div>' if score is not None else ""
    return (f'<div style="width:122px;margin:4px;font-family:system-ui,sans-serif;text-align:center">'
            f'{head}{pic}'
            f'<div style="font-size:9px;color:#333;margin-top:2px">{_html.escape(asin)}</div>'
            f'{sc}</div>')


def compare_ttn(query_asin: str):
    """Side-by-side look at both TTN recommendation approaches for one item,
    as product pictures rather than bare asin codes."""
    flat_rows = ttn_flat[ttn_flat["query_asin"] == query_asin]
    prop_rows = ttn_proportional[ttn_proportional["query_asin"] == query_asin]
    if flat_rows.empty and prop_rows.empty:
        display(HTML(f"<p>{_html.escape(query_asin)} has no TTN recommendations "
                      f"in either file.</p>"))
        return

    query_card = _card(query_asin)

    flat_cards = "".join(
        _card(r.candidate_asin, score=r.score) for r in flat_rows.itertuples()
    )

    # Group the proportional rows by target_node_id so each category's slots
    # are visually set apart, labelling only the first card in each group.
    prop_cards = []
    for node_id, grp in prop_rows.groupby("target_node_id", sort=False):
        for i, r in enumerate(grp.itertuples()):
            caption = f"category {node_id}" if i == 0 else ""
            prop_cards.append(_card(r.candidate_asin, caption=caption, score=r.score))
    prop_cards = "".join(prop_cards)

    display(HTML(
        '<div style="font-family:system-ui,sans-serif">'
        '<div style="font-size:13px;font-weight:600;margin-bottom:4px">QUERY</div>'
        f'<div style="display:flex">{query_card}</div>'

        '<div style="font-size:13px;font-weight:600;margin:12px 0 4px">'
        'COMPLEMENTS -- flat merge (complements.parquet)</div>'
        f'<div style="display:flex;flex-wrap:wrap">{flat_cards}</div>'

        '<div style="font-size:13px;font-weight:600;margin:12px 0 4px">'
        f'COMPLEMENTS -- proportional by category evidence '
        f'(complements_proportional.parquet, {prop_rows["target_node_id"].nunique()} categories)</div>'
        f'<div style="display:flex;flex-wrap:wrap">{prop_cards}</div>'
        '</div>'
    ))


# Pick a sample item that actually has recommendations in both files -- the
# same one used earlier this session to demonstrate the flat merge's
# single-category-domination problem.
SAMPLE_ASIN = "0560467893"
compare_ttn(SAMPLE_ASIN)

## Popularity, for the same input item

Popularity is keyed by **category**, not by individual item -- every item
sharing `SAMPLE_ASIN`'s own category sees this exact same list (see
`Popularity/README.md`). So "results for the same input item" here means
"the list for whichever category that item's own product belongs to," in
both variants (`all_time`, `recency`).

In [ ]:
# Popularity no longer uses TTN's co-purchase-derived node_of_item.npy --
# it reads each item's own (cat_2, cat_3, cat_4) straight from
# df_features.pkl now (see Popularity/build_popularity.py's docstring for
# why). Reuses image_by_asin/_card from the TTN cell above; doesn't reuse
# df_features itself since that cell never loaded the full table (it
# loads the small image-url cache instead).
pop_cat_by_asin = (pd.read_pickle(ROOT / "data" / "df_features.pkl")[["asin", "cat_2", "cat_3", "cat_4"]]
                   .drop_duplicates("asin").set_index("asin"))
pop_cat_by_asin["cat_4"] = pop_cat_by_asin["cat_4"].fillna("Missing")

popularity = pd.read_parquet(popularity_path)


def show_popularity(query_asin: str):
    """Picture cards for the Popularity carousel, both variants, for
    whichever category query_asin's own item belongs to -- reuses the same
    _card() helper defined in the TTN comparison cell above."""
    if query_asin not in pop_cat_by_asin.index:
        display(HTML(f"<p>{_html.escape(query_asin)} has no category in "
                      f"df_features.pkl.</p>"))
        return
    c2, c3, c4 = pop_cat_by_asin.loc[query_asin, ["cat_2", "cat_3", "cat_4"]]
    rows = popularity[(popularity["cat_2"] == c2) & (popularity["cat_3"] == c3)
                       & (popularity["cat_4"] == c4)]
    if rows.empty:
        display(HTML(f"<p>no popularity rows for category "
                      f"{_html.escape(f'{c2} > {c3} > {c4}')} "
                      f"({_html.escape(query_asin)}'s own category).</p>"))
        return

    sections = [
        '<div style="font-size:13px;font-weight:600;margin-bottom:4px">QUERY</div>'
        f'<div style="display:flex">{_card(query_asin)}</div>'
        f'<div style="font-size:11px;color:#888;margin:4px 0 8px">'
        f'own category: {_html.escape(f"{c2} > {c3} > {c4}")} -- every item in '
        f'this category sees the identical list below, regardless of which one is asked</div>'
    ]
    for variant, grp in rows.groupby("variant"):
        cards = "".join(_card(r.candidate_asin, score=r.score)
                         for r in grp.sort_values("rank").itertuples())
        sections.append(
            f'<div style="font-size:13px;font-weight:600;margin:12px 0 4px">'
            f'POPULAR -- {variant}</div>'
            f'<div style="display:flex;flex-wrap:wrap">{cards}</div>'
        )

    display(HTML('<div style="font-family:system-ui,sans-serif">' + "".join(sections) + '</div>'))


show_popularity(SAMPLE_ASIN)

## SigLIP2 -- adapted vs. raw "visually similar" embeddings

The SigLIP2 champion is the imported `pilot_200k` adapter -- but until now,
nothing had ever run it across a full snapshot and saved the result; it
had only been verified on individual items. `SigLIP2/generate_adapted_image_embeddings.py`
closes that gap, applying the champion adapter to this snapshot's existing
raw image embeddings catalogue-wide (`SigLIP2/generated/<snapshot>/adapted_img_emb.npy`).

Scope, by deliberate choice made earlier this session: **image-to-image
only** -- this upgrades the existing "visually similar" feature with
adapted embeddings instead of raw ones. It is *not* cross-modal
(text-to-image) search, which would need raw SigLIP2 text embeddings that
don't exist at this scale yet.

The cell below shows both versions side by side, same query item, same
in-node candidate pool (the same discipline `SigLIP2/generate_recommendations.py`
already uses) -- only the embedding each one ranks by differs.

In [ ]:
# SigLIP2's champion has no snapshot_id of its own (checkpoints/pilot_200k/
# isn't tied to any --window-days snapshot -- see champion_selection_siglip.json's
# own note). So this reuses ttn_snapshot purely because every champion in
# this notebook happens to point at the same snapshot today, not because
# SigLIP2's champion says to.
siglip_snapshot = ttn_snapshot
siglip_dir = ROOT / "SigLIP2" / "generated" / siglip_snapshot

raw_img_path = siglip_dir / "siglip_img_emb.npy"
adapted_img_path = siglip_dir / "adapted_img_emb.npy"
assert adapted_img_path.exists(), (
    f"missing: {adapted_img_path} -- run "
    f"'python SigLIP2/generate_adapted_image_embeddings.py --snapshot {siglip_snapshot}' first")

raw_img_emb = np.load(raw_img_path).astype("float32")
adapted_img_emb = np.load(adapted_img_path).astype("float32")
img_status = np.load(siglip_dir / "siglip_img_status.npy")
has_img = img_status == 1

siglip_asins = np.load(ROOT / "data" / "tower" / siglip_snapshot / "item_asins.npy",
                        allow_pickle=False).astype(str)
siglip_node_of_item = np.load(ROOT / "data" / "tower" / siglip_snapshot / "node_of_item.npy")
siglip_idx_of = {a: i for i, a in enumerate(siglip_asins)}

SIGLIP_TOP_K = 10


def _top_k_visually_similar(emb: np.ndarray, q_idx: int, cand: np.ndarray, k: int = SIGLIP_TOP_K):
    """Cosine top-k within a fixed candidate pool -- same in-node discipline
    SigLIP2/generate_recommendations.py already uses, just inlined here."""
    qn = emb[q_idx] / (np.linalg.norm(emb[q_idx]) + 1e-9)
    cn = emb[cand] / (np.linalg.norm(emb[cand], axis=1, keepdims=True) + 1e-9)
    scores = cn @ qn
    top = np.argsort(-scores)[:k]
    return cand[top], scores[top]


def compare_siglip(query_asin: str):
    """Side-by-side image-to-image results: raw vs. champion-adapted
    embeddings, same query, same candidate pool."""
    if query_asin not in siglip_idx_of:
        display(HTML(f"<p>{_html.escape(query_asin)} not found in "
                      f"{siglip_snapshot}'s item list.</p>"))
        return
    q_idx = siglip_idx_of[query_asin]
    if not has_img[q_idx]:
        display(HTML(f"<p>{_html.escape(query_asin)} has no usable image "
                      f"-- SigLIP2 can't rank it.</p>"))
        return

    node_id = siglip_node_of_item[q_idx]
    cand = np.where((siglip_node_of_item == node_id) & has_img)[0]
    cand = cand[cand != q_idx]
    if len(cand) == 0:
        display(HTML(f"<p>no other items with an image share "
                      f"{_html.escape(query_asin)}'s category.</p>"))
        return

    raw_top, raw_scores = _top_k_visually_similar(raw_img_emb, q_idx, cand)
    adapted_top, adapted_scores = _top_k_visually_similar(adapted_img_emb, q_idx, cand)
    overlap = len(set(raw_top.tolist()) & set(adapted_top.tolist()))

    raw_cards = "".join(_card(siglip_asins[i], score=float(s))
                         for i, s in zip(raw_top, raw_scores))
    adapted_cards = "".join(_card(siglip_asins[i], score=float(s))
                             for i, s in zip(adapted_top, adapted_scores))

    display(HTML(
        '<div style="font-family:system-ui,sans-serif">'
        '<div style="font-size:13px;font-weight:600;margin-bottom:4px">QUERY</div>'
        f'<div style="display:flex">{_card(query_asin)}</div>'
        f'<div style="font-size:11px;color:#888;margin:4px 0 8px">'
        f'{len(cand)} candidates in this category with a usable image -- '
        f'top-{SIGLIP_TOP_K} overlap between raw and adapted: {overlap}/{SIGLIP_TOP_K}</div>'

        '<div style="font-size:13px;font-weight:600;margin:12px 0 4px">'
        'VISUALLY SIMILAR -- raw embeddings (siglip_img_emb.npy)</div>'
        f'<div style="display:flex;flex-wrap:wrap">{raw_cards}</div>'

        '<div style="font-size:13px;font-weight:600;margin:12px 0 4px">'
        'VISUALLY SIMILAR -- champion-adapted embeddings (adapted_img_emb.npy)</div>'
        f'<div style="display:flex;flex-wrap:wrap">{adapted_cards}</div>'
        '</div>'
    ))


compare_siglip(SAMPLE_ASIN)